In [1]:
!git clone https://github.com/bremsstrahlung-57/practicum-project.git

Cloning into 'practicum-project'...
remote: Enumerating objects: 38, done.
remote: Counting objects: 100% (3/3), done.
remote: Compressing objects: 100% (2/2), done.
remote: Total 38 (delta 2), reused 1 (delta 1), pack-reused 35 (from 1)
Receiving objects: 100% (38/38), 132.66 MiB | 21.00 MiB/s, done.
Resolving deltas: 100% (10/10), done.
Updating files: 100% (9/9), done.


In [2]:
from google.colab import userdata
WANDB_API_KEY = userdata.get('WANDB_API_KEY')
import wandb
wandb.login(WANDB_API_KEY)

/usr/local/lib/python3.12/dist-packages/notebook/notebookapp.py:191: SyntaxWarning: invalid escape sequence '\/'
  | |_| | '_ \/ _` / _` |  _/ -_)
wandb: WARNING If you're specifying your api key in code, ensure this code is not shared publicly.
wandb: WARNING Consider setting the WANDB_API_KEY environment variable, or running `wandb login` from the command line.
wandb: [wandb.login()] Using explicit session credentials for https://api.wandb.ai.
wandb: No netrc file found, creating one.
wandb: Appending key for api.wandb.ai to your netrc file: /root/.netrc
wandb: Currently logged in as: sagar-sharma-03015611924 (practicum-project) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


True

In [5]:
import torch
import torch.nn as nn
import copy
import time
import tracemalloc
import os
import torchvision
import numpy as np

In [6]:
WARMUP_RUNS = 20
TIMING_RUNS = 100
DEVICE      = 'cpu'

In [7]:
def ResNet18CIFAR():
    model = torchvision.models.resnet18(weights=None)
    model.conv1 = torch.nn.Conv2d(3, 64, kernel_size=3, stride=1, padding=1, bias=False)
    model.maxpool = torch.nn.Identity()
    model.fc = torch.nn.Linear(512, 10)
    return model

# we are using 70% pruned model
model = ResNet18CIFAR()
model.load_state_dict(torch.load('/content/practicum-project/pruned/pruned_70.pth', map_location='cpu'))

<All keys matched successfully>

In [8]:
model_dynamic = torch.quantization.quantize_dynamic(
    copy.deepcopy(model).to(DEVICE),
    {nn.Conv2d, nn.Linear},
    dtype=torch.qint8
)
model_dynamic.eval()
print("Dynamic INT8 model ready.")

Dynamic INT8 model ready.


/tmp/ipykernel_1243/3462958270.py:1: DeprecationWarning: torch.ao.quantization is deprecated and will be removed in 2.10. 
For migrations of users: 
1. Eager mode quantization (torch.ao.quantization.quantize, torch.ao.quantization.quantize_dynamic), please migrate to use torchao eager mode quantize_ API instead 
2. FX graph mode quantization (torch.ao.quantization.quantize_fx.prepare_fx,torch.ao.quantization.quantize_fx.convert_fx, please migrate to use torchao pt2e quantization API instead (prepare_pt2e, convert_pt2e) 
3. pt2e quantization has been migrated to torchao (https://github.com/pytorch/ao/tree/main/torchao/quantization/pt2e) 
see https://github.com/pytorch/ao/issues/2259 for more details
  model_dynamic = torch.quantization.quantize_dynamic(


In [9]:
def evaluate(net, loader, device='cpu'):
    net.eval()
    correct, total = 0, 0
    with torch.no_grad():
        for images, labels in loader:
            images, labels = images.to(device), labels.to(device)
            outputs = net(images)
            _, predicted = outputs.max(1)
            correct += predicted.eq(labels).sum().item()
            total += labels.size(0)
    return 100.0 * correct / total

In [10]:
def get_model_size_mb(net, path='tmp_model.pth'):
    torch.save(net.state_dict(), path)
    size_mb = os.path.getsize(path) / 1e6
    os.remove(path)
    return size_mb

In [11]:
def benchmark_latency(net, warmup=WARMUP_RUNS, runs=TIMING_RUNS, device='cpu'):
    net.eval()
    dummy = torch.randn(1, 3, 32, 32).to(device)
    with torch.no_grad():
        for _ in range(warmup):
            net(dummy)
    times = []
    with torch.no_grad():
        for _ in range(runs):
            t0 = time.perf_counter()
            net(dummy)
            times.append((time.perf_counter() - t0) * 1000)
    return {'mean_ms': sum(times)/len(times), 'min_ms': min(times), 'max_ms': max(times)}

In [12]:
def benchmark_ram(net, device='cpu'):
    net.eval()
    dummy = torch.randn(1, 3, 32, 32).to(device)
    tracemalloc.start()
    with torch.no_grad():
        net(dummy)
    _, peak = tracemalloc.get_traced_memory()
    tracemalloc.stop()
    return peak / 1e6

In [13]:
import torchvision.transforms as transforms

transform_train = transforms.Compose([
    transforms.RandomCrop(32, padding=4),
    transforms.RandomHorizontalFlip(),
    transforms.ToTensor(),
    transforms.Normalize((0.4914, 0.4822, 0.4465),
                         (0.2023, 0.1994, 0.2010)),
])

transform_test = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.4914, 0.4822, 0.4465),
                         (0.2023, 0.1994, 0.2010)),
])

trainset = torchvision.datasets.CIFAR10(root='./data', train=True,
                                         download=True, transform=transform_train)
testset  = torchvision.datasets.CIFAR10(root='./data', train=False,
                                         download=True, transform=transform_test)
trainloader = torch.utils.data.DataLoader(trainset, batch_size=128, shuffle=True,
                         num_workers=2, pin_memory=True)
testloader  = torch.utils.data.DataLoader(testset,  batch_size=128, shuffle=False,
                         num_workers=2, pin_memory=True)


100%|██████████| 170M/170M [00:01<00:00, 100MB/s]


In [14]:
acc_dynamic = evaluate(model_dynamic, testloader, device=DEVICE)
print(f"Dynamic INT8 Accuracy: {acc_dynamic:.2f}%")

size_fp32    = get_model_size_mb(model)
size_dynamic = get_model_size_mb(model_dynamic)
print(f"FP32: {size_fp32:.2f} MB  |  Dynamic INT8: {size_dynamic:.2f} MB")
print(f"Compression: {size_fp32 / size_dynamic:.2f}x")

lat_fp32    = benchmark_latency(model.to(DEVICE))
lat_dynamic = benchmark_latency(model_dynamic)
print(f"Latency FP32:    {lat_fp32['mean_ms']:.3f} ms")
print(f"Latency Dynamic: {lat_dynamic['mean_ms']:.3f} ms")
print(f"Speedup: {lat_fp32['mean_ms'] / lat_dynamic['mean_ms']:.2f}x")

ram_fp32    = benchmark_ram(model.to(DEVICE))
ram_dynamic = benchmark_ram(model_dynamic)
print(f"Peak RAM FP32:    {ram_fp32:.2f} MB")
print(f"Peak RAM Dynamic: {ram_dynamic:.2f} MB")

wandb.init(project="cnn-compression", name="quantization-dynamic-int8", reinit=True)
wandb.log({
    "technique":        "dynamic_int8",
    "accuracy":         acc_dynamic,
    "model_size_mb":    size_dynamic,
    "latency_mean_ms":  lat_dynamic['mean_ms'],
    "latency_min_ms":   lat_dynamic['min_ms'],
    "latency_max_ms":   lat_dynamic['max_ms'],
    "ram_peak_mb":      ram_dynamic,
    "size_reduction_x": round(size_fp32 / size_dynamic, 2),
    "speedup_x":        round(lat_fp32['mean_ms'] / lat_dynamic['mean_ms'], 2),
    "acc_drop":         round(acc_dynamic - 95.34, 2),
})
wandb.finish()

print("\n── Summary ──────────────────────────")
print(f"Accuracy drop : {acc_dynamic - 95.34:+.2f}%")
print(f"Size reduction: {size_fp32 / size_dynamic:.2f}x")
print(f"Speedup       : {lat_fp32['mean_ms'] / lat_dynamic['mean_ms']:.2f}x")
print(f"RAM change    : {ram_fp32 / ram_dynamic:.2f}x")

/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:1118: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


Dynamic INT8 Accuracy: 94.81%
FP32: 44.77 MB  |  Dynamic INT8: 44.76 MB
Compression: 1.00x
Latency FP32:    28.208 ms
Latency Dynamic: 28.182 ms
Speedup: 1.00x
Peak RAM FP32:    0.00 MB
Peak RAM Dynamic: 0.00 MB


wandb: WARNING Using a boolean value for 'reinit' is deprecated. Use 'return_previous' or 'finish_previous' instead.


acc_drop,▁
accuracy,▁
latency_max_ms,▁
latency_mean_ms,▁
latency_min_ms,▁
model_size_mb,▁
ram_peak_mb,▁
size_reduction_x,▁
speedup_x,▁
acc_drop,-0.53
accuracy,94.81



── Summary ──────────────────────────
Accuracy drop : -0.53%
Size reduction: 1.00x
Speedup       : 1.00x
RAM change    : 1.00x


In [15]:
torch.save(model_dynamic.state_dict(), 'resnet18_dynamic_int8.pth')
print("Saved.")

Saved.
